In [2]:
import numpy as np
import torch

In [ ]:
import torch


def cell_areas(
    dx: torch.Tensor,          # (N, m, 3) normalized neighbor directions
    d: torch.Tensor,           # (N, m) neighbor distances
    p: torch.Tensor,           # (N, 3) apicobasal normal (unit)
    q: torch.Tensor,           # (N, 3) PCP direction (unit, tangent)
    neighbor_mask: torch.Tensor # (N, m) True = invalid neighbor
):
    """
    Returns:
        A : (N,) approximate Voronoi-like cell areas
    """

    N, m, _ = dx.shape
    device = dx.device
    eps = 1e-8

    # ------------------------------------------------------------------
    # 1. Tangent basis per cell
    # ------------------------------------------------------------------
    t1 = q                                                # (N, 3)
    t2 = torch.cross(p, q, dim=1)                         # (N, 3)
    t2 = t2 / (torch.norm(t2, dim=1, keepdim=True) + eps)

    # ------------------------------------------------------------------
    # 2. Displacement vectors r_ij
    # ------------------------------------------------------------------
    r = dx * d[..., None]                                 # (N, m, 3)

    # ------------------------------------------------------------------
    # 3. Project into local tangent plane
    # ------------------------------------------------------------------
    x = torch.einsum("nmk,nk->nm", r, t1)                  # (N, m)
    y = torch.einsum("nmk,nk->nm", r, t2)                  # (N, m)

    # ------------------------------------------------------------------
    # 4. Compute angles and sort neighbors cyclically
    # ------------------------------------------------------------------
    angles = torch.atan2(y, x)                            # (N, m)
    angles = angles.masked_fill(neighbor_mask, float("inf"))

    order = torch.argsort(angles, dim=1)                  # (N, m)

    x = torch.gather(x, 1, order)
    y = torch.gather(y, 1, order)
    valid = ~torch.gather(neighbor_mask, 1, order)        # (N, m)

    # ------------------------------------------------------------------
    # 5. Shoelace formula (open edges)
    # ------------------------------------------------------------------
    x_next = torch.roll(x, shifts=-1, dims=1)
    y_next = torch.roll(y, shifts=-1, dims=1)
    valid_next = torch.roll(valid, shifts=-1, dims=1)

    edge_valid = valid & valid_next

    cross = (x * y_next - x_next * y) * edge_valid        # (N, m)
    cross_sum = cross.sum(dim=1)                           # (N,)

    # ------------------------------------------------------------------
    # 6. Explicitly close the polygon (last valid → first valid)
    # ------------------------------------------------------------------
    k = valid.sum(dim=1)                                   # (N,)
    has_polygon = k >= 3

    last = (k - 1).clamp(min=0)                            # (N,)
    idx = torch.arange(N, device=device)

    x_last = x[idx, last]
    y_last = y[idx, last]
    x_first = x[idx, 0]
    y_first = y[idx, 0]

    closing = x_last * y_first - x_first * y_last
    closing = closing * has_polygon                        # zero if <3 neighbors

    # ------------------------------------------------------------------
    # 7. Final area
    # ------------------------------------------------------------------
    A_dual = 0.5 * torch.abs(cross_sum + closing)

    # Convert dual area → Voronoi-like area
    A = A_dual / 3.0

    return A

In [ ]:
def cell_areas(dx, d, p, q, neighbor_mask):
    # tangent basis
    t1 = q
    t2 = torch.cross(p, q, dim=1)
    t2 = t2 / (torch.norm(t2, dim=1, keepdim=True) + 1e-8)

    # displacements
    r = dx * d[..., None]

    # project
    x = torch.einsum('nmk,nk->nm', r, t1)
    y = torch.einsum('nmk,nk->nm', r, t2)

    # mask invalid neighbors
    large = 1e6
    x = x.masked_fill(neighbor_mask, large)
    y = y.masked_fill(neighbor_mask, large)

    # angles and ordering
    angles = torch.atan2(y, x)
    order = torch.argsort(angles, dim=1)

    x = torch.gather(x, 1, order)
    y = torch.gather(y, 1, order)
    valid = ~neighbor_mask.gather(1, order)

    # shoelace
    x_next = torch.roll(x, -1, dims=1)
    y_next = torch.roll(y, -1, dims=1)
    cross = (x * y_next - x_next * y) * valid

    A_dual = 0.5 * torch.abs(cross.sum(dim=1))
    return A_dual / 3.0

In [24]:
import numpy as np
from scipy.optimize import brentq

d0 = 5*np.log(5) / 4

def d0_i(a, b, xmax=None):
    """
    Compute the x that minimizes
    f(x) = exp(-a x) - exp(-a x/5) + exp(-b x) - exp(-b x/5)

    Parameters
    ----------
    a, b : positive floats
    xmax : optional float, upper search bound

    Returns
    -------
    x0 : float
        Location of the unique minimum
    """

    a = 1/a
    b = 1/b
    
    if a <= 0 or b <= 0:
        raise ValueError("a and b must be positive")

    def df(x):
        return (
            -a * np.exp(-a * x) + (a / 5) * np.exp(-a * x / 5)
            -b * np.exp(-b * x) + (b / 5) * np.exp(-b * x / 5)
        )

    # f'(0) < 0 always
    x_lo = 0.0

    # Sensible upper bound if none provided
    if xmax is None:
        xmax = 10.0 / min(a, b)

    x_hi = xmax

    # Expand upper bound until sign change is guaranteed
    while df(x_hi) < 0:
        x_hi *= 2
        if x_hi > 1e6:
            raise RuntimeError("Failed to bracket root")

    return brentq(df, x_lo, x_hi)

def d0_mean(a, b):
    return d0 * (a + b) / 2 

def d0_geomean(a, b):
    return d0 * np.sqrt(a * b) 


In [28]:
print(d0_mean(1.0, 2))

print(d0_i(1.0, 2))
print(d0_geomean(1.0, 2))

3.017696085813938
2.7396883074265945
2.845111154452183


In [53]:
tens = torch.tensor([[1,1,1,1],[2,2,2,2],[3,3,3,3]])
tens = tens[:,:,None].expand(3,4,3)
trans_tens = torch.transpose_copy(tens, 0,1)
print(tens[0])
print(trans_tens[0])
# print((trans_tens + tens) /2)

tensor([[1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1]])
tensor([[1, 1, 1],
        [2, 2, 2],
        [3, 3, 3]])


In [5]:
data = np.load('moar_gamma_cos_2/data.pkl', allow_pickle=True)

In [12]:
x = data['x']

print(x[-1])

[[nan nan nan]
 [nan nan nan]
 [nan nan nan]
 ...
 [nan nan nan]
 [nan nan nan]
 [nan nan nan]]


In [ ]:
dot = torch.abs(torch.einsum('ijk,ijk->ij', dx, qi,)) # Shape: [n_cells, max_neighbors]


In [ ]:
elongated = self.elongatable[:,None] & self.elongatable[idx]

dot = torch.abs(torch.einsum('ijk,ijk->ij', dx, elonq,)) # Shape: [n_cells, max_neighbors]
angle = torch.arccos(dot)/np.pi

scaled_angle = 2 * angle                                 # 1 when parallel, 0 when perpendicular
elon_factor = 1. - (1 - 2.*scaled_angle) * gammas
elon_factor = torch.where(elongated,  elon_factor, torch.tensor(1.0))[:,:,None]

d_tilde = (d[:,:,None] * elon_factor).squeeze(2)
    



In [ ]:
def find_knn_torch_unchuncked(self, x, k=100):
    with torch.no_grad():
        dists = torch.cdist(x, x)
        knn_indices = dists.topk(k + 1, largest=False).indices[:, 1:]
        knn_dists = dists[torch.arange(dists.shape[0])[:, None], knn_indices]
        return knn_dists, knn_indices

    with torch.no_grad():
        z_mask = torch.tensor(d < 4.5, dtype = bool, device = self.device)


In [9]:
lst = [1, 2, np.array([3,4]), 1, np.array([5,6,7,8])]

lst_copy = lst.copy()
first_elem = lst_copy.pop(0)
lst_copy.append(first_elem)

print(lst_copy)
print(lst)

[2, array([3, 4]), 1, array([5, 6, 7, 8]), 1]
[1, 2, array([3, 4]), 1, array([5, 6, 7, 8])]


In [12]:
lst_copy = lst.copy()
lst_copy = lst_copy[1:] + [lst_copy[0]]

print(lst_copy)

[2, array([3, 4]), 1, array([5, 6, 7, 8]), 1]
